In [1]:
#install libaries
import os
import platform
import numpy as np
import pandas as pd
from google import genai
from google.genai import types
from dotenv import load_dotenv
from sklearn.metrics.pairwise import cosine_similarity
import time
from transformers import AutoTokenizer, AutoModel
import torch
from sentence_transformers import SentenceTransformer
%pip install faiss-cpu
import faiss
from scipy.special import softmax


/Users/daisy/anaconda3/envs/search-engine-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 227.1 kB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [8]:
#api keys
# Check if we are in Google Colab
if platform.system() == 'Linux' and 'colab' in os.path.basename(os.getcwd()).lower():
    # This means we are in Google Colab
    dotenv_path = '/content/.env'  # Colab specific path
else:
    # This is for local VS Code
    dotenv_path = '.env'  # Local path to .env

load_dotenv(dotenv_path)
google_api_key = os.getenv("GOOGLE_API_KEY")
client = genai.Client(api_key=google_api_key)

In [7]:
#fixture data
user_queries = [
    "batman comics from the 80s",
    "first edition pokemon cards",
    "vintage vinyl jazz albums",
    "antique silver coins pre 1900",
    "limited edition hot wheels cars",
    "signed baseball memorabilia",
    "star wars action figures 1977",
    "graded magic the gathering cards",
    "new condition barbie dolls",
    "beatles abbey road vinyl"
]

product_listings = [
    {
        "title": "Batman #404 Year One Part 1 1987",
        "description": "Classic Batman comic by Frank Miller, first issue in the 'Year One' storyline. Near mint condition, perfect for DC collectors."
    },
    {
        "title": "Charizard Holo 1st Edition",
        "description": "Iconic Pokémon card from the Base Set. Highly collectible, graded PSA 9 for condition. A grail for any TCG fan."
    },
    {
        "title": "Miles Davis Kind of Blue 1959 Vinyl",
        "description": "Vintage Columbia 6-eye pressing of the legendary jazz album. Slight wear on cover, vinyl is VG+."
    },
    {
        "title": "1904 Morgan Silver Dollar",
        "description": "Beautiful example of a Morgan Dollar, Philadelphia Mint. Uncirculated with high detail on Liberty's face."
    },
    {
        "title": "Hot Wheels 1995 Treasure Hunt '57 Chevy",
        "description": "Rare 1995 Treasure Hunt series car with original packaging. Highly sought-after piece from the early TH lineup."
    },
    {
        "title": "Derek Jeter Signed Yankees Jersey",
        "description": "Official MLB jersey signed by Derek Jeter with COA. Great addition to any sports memorabilia collection."
    },
    {
        "title": "Kenner Star Wars Luke Skywalker (1978)",
        "description": "Original action figure with telescoping lightsaber. Shows minor wear, but great articulation and paint detail."
    },
    {
        "title": "MTG Black Lotus Proxy for Display",
        "description": "High quality display proxy of the Black Lotus card from Alpha set. For aesthetic purposes only, not tournament legal."
    },
    {
        "title": "Barbie 1985 Day to Night Edition",
        "description": "Vintage Barbie in both business and evening outfits. Excellent condition with original box."
    },
    {
        "title": "The Beatles Abbey Road (1969 UK Pressing)",
        "description": "Original UK pressing with Apple label. Minor sleeve wear, but vinyl plays beautifully. Essential for any Beatles collection."
    }
]


**Gemini Embedding Experimental**

Experiemental, ranked first in Hugging Face MTEB leaderboard. Last updated in March 2025

Product listing (title + description), task type: RETRIEVAL_DOCUMENT

User query (search), task type: RETRIEVAL_QUERY

Requests Per Minute: 5
Requests Per Day: 100

In [34]:
#test
result1 = client.models.embed_content(
        model="gemini-embedding-exp-03-07",
        contents="How does alphafold work?",
)

#print(result1.embeddings[:1])

#similarilty serach - Retrieval Augmented Generation (RAG)
result2 = client.models.embed_content(
        model="gemini-embedding-exp-03-07",
        contents="What is the meaning of life?",
        config=types.EmbedContentConfig(task_type="SEMANTIC_SIMILARITY")
)
#print(result2.embeddings[:1])

In [4]:
def embed_text(text, task_type):
    response = client.models.embed_content(
        model="gemini-embedding-exp-03-07",
        contents=text,
        config=types.EmbedContentConfig(task_type=task_type)
    )
    return response.embeddings

In [9]:
# embed the product listings and the user queries
product_embeddings = []
for item in product_listings:
    text = f"{item['title']} {item['description']}"
    vec = embed_text(text, task_type="RETRIEVAL_DOCUMENT")
    if vec:
        product_embeddings.append(vec)
    time.sleep(15)

query_embeddings = []
for query in user_queries:
    vec = embed_text(query, task_type="RETRIEVAL_QUERY")
    if vec:
        query_embeddings.append(vec)
    time.sleep(15)

In [10]:
#print(product_embeddings)
#print(product_embeddings[0])
#print(product_embeddings[0][0].values)
print(np.array(product_embeddings).shape)
print(np.array(query_embeddings).shape)

(10, 1)
(10, 1)


In [11]:
product_embeddings_np = np.array([item[0].values for item in product_embeddings])  # Extract the 'values' from each embedding object
query_embeddings_np = np.array([query_vec[0].values for query_vec in query_embeddings])  # Extract the 'values' from each query embedding

for i, query_vec in enumerate(query_embeddings_np):
    query_vec = query_vec.reshape(1, -1)  # Shape: (1, D)
    scores = cosine_similarity(query_vec, product_embeddings_np)[0]  # Shape: (N,)
    top_indices = scores.argsort()[::-1][:3]

    print(f"\nQuery: {user_queries[i]}")
    for rank, idx in enumerate(top_indices, start=1):
        product = product_listings[idx]
        print(f"{rank}. {product['title']} (score: {scores[idx]:.4f} - {(scores[idx] * 100):.2f}%)")


Query: batman comics from the 80s
1. Batman #404 Year One Part 1 1987 (score: 0.7160 - 71.60%)
2. Barbie 1985 Day to Night Edition (score: 0.5692 - 56.92%)
3. Charizard Holo 1st Edition (score: 0.5540 - 55.40%)

Query: first edition pokemon cards
1. Charizard Holo 1st Edition (score: 0.7274 - 72.74%)
2. MTG Black Lotus Proxy for Display (score: 0.6338 - 63.38%)
3. Hot Wheels 1995 Treasure Hunt '57 Chevy (score: 0.6068 - 60.68%)

Query: vintage vinyl jazz albums
1. Miles Davis Kind of Blue 1959 Vinyl (score: 0.7261 - 72.61%)
2. The Beatles Abbey Road (1969 UK Pressing) (score: 0.6410 - 64.10%)
3. Barbie 1985 Day to Night Edition (score: 0.5820 - 58.20%)

Query: antique silver coins pre 1900
1. 1904 Morgan Silver Dollar (score: 0.6801 - 68.01%)
2. Hot Wheels 1995 Treasure Hunt '57 Chevy (score: 0.5786 - 57.86%)
3. Miles Davis Kind of Blue 1959 Vinyl (score: 0.5741 - 57.41%)

Query: limited edition hot wheels cars
1. Hot Wheels 1995 Treasure Hunt '57 Chevy (score: 0.6977 - 69.77%)
2. Bar

**Linq-AI-Research/Linq-Embed-Mistral**

Ranked 2nd in the MTEB. Developed on  E5-mistral-7b-instruct and Mistral-7B-v0.1 models. Focus on imporoving text retrieval.

Open source

In [ ]:
# sentence-transformers library - too big to run locally
st_model = SentenceTransformer("Linq-AI-Research/Linq-Embed-Mistral")

In [ ]:
#continuing with sentence-transformers library, encode user queries & product listisngs
product_texts = [f"{item['title']} {item['description']}" for item in product_listings]
product_embeddings_lem_st = st_model.encode(product_texts, convert_to_numpy=True)

query_embeddings_lem_st = st_model.encode(user_queries, convert_to_numpy=True)

In [ ]:
# compare the vectors

In [ ]:
# transformer libary
tokenizer = AutoTokenizer.from_pretrained("Linq-AI-Research/Linq-Embed-Mistral")
model = AutoModel.from_pretrained("Linq-AI-Research/Linq-Embed-Mistral")

Loading checkpoint shards: 100%|██████████| 3/3 [01:06<00:00, 22.08s/it]


In [ ]:
# transformer library
def embed_text_LEM(text):
    inputs = tokenizer(        # converts raw text into model ready format
        text,
        return_tensors="pt",   # return PyTorch tensors
        truncation=True,       # Make sure long text gets clipped
        max_length=512         # You can increase this if needed (1024?)
    )

    # disables gradient calculations for faster inference
    with torch.no_grad():
        outputs = model(**inputs) # feeds the tokenized inputs into the model, outputs last_hidden_state
                                  # which is a tensor of shape

    print(outputs)

    # This assumes the embedding is in the last_hidden_state and you average it
    return outputs.last_hidden_state.mean(dim=1).squeeze().numpy()

In [ ]:
# transformer library
query_embeddings_LEM = [embed_text_LEM(query) for query in user_queries]
product_embeddings_LEM = [embed_text_LEM(f"{item['title']} {item['description']}") for item in product_listings]

In [ ]:
for i, query_vec in enumerate(query_embeddings_LEM):
    query_vec = query_vec.reshape(1, -1)
    product_matrix = np.array(product_embeddings)
    scores = cosine_similarity(query_vec, product_matrix)[0]
    top_indices = scores.argsort()[::-1][:3]

    print(f"\nQuery: {user_queries[i]}")
    for rank, idx in enumerate(top_indices, start=1):
        product = product_listings[idx]
        print(f"{rank}. {product['title']} (score: {scores[idx]:.4f})")

**FAISS**

Allows developers to quickly search for embeddings of multimedia documentse that are similar to each other.

As it's a vector index, using it in this case to search for vectors.

In [12]:
# convert the product & user embeddings
gemini_product_vec = np.array(
    [np.array(embed[0].values, dtype='float32') for embed in product_embeddings]
    )

gemini_query_vec = np.array(
    [np.array(embed[0].values, dtype='float32') for embed in query_embeddings]
    )

In [13]:
#normalize for cosine similarity & build faiss index
faiss.normalize_L2(gemini_product_vec)

index = faiss.IndexFlatIP(gemini_product_vec.shape[1])
index.add(gemini_product_vec)

faiss.normalize_L2(gemini_query_vec)

In [16]:
# search for each query
for i, qv in enumerate(gemini_query_vec):
    D, I = index.search(qv.reshape(1, -1), k=3)  # top 3 results

    print(f"\nQuery: {user_queries[i]}")
    for rank, idx in enumerate(I[0]):
        score = D[0][rank]
        product = product_listings[idx]
        print(f"{rank + 1}. {product['title']} (score: {score:.4f} - {(score * 100):.2f}%)")


Query: batman comics from the 80s
1. Batman #404 Year One Part 1 1987 (score: 0.7160 - 71.60%)
2. Barbie 1985 Day to Night Edition (score: 0.5692 - 56.92%)
3. Charizard Holo 1st Edition (score: 0.5540 - 55.40%)

Query: first edition pokemon cards
1. Charizard Holo 1st Edition (score: 0.7274 - 72.74%)
2. MTG Black Lotus Proxy for Display (score: 0.6338 - 63.38%)
3. Hot Wheels 1995 Treasure Hunt '57 Chevy (score: 0.6068 - 60.68%)

Query: vintage vinyl jazz albums
1. Miles Davis Kind of Blue 1959 Vinyl (score: 0.7261 - 72.61%)
2. The Beatles Abbey Road (1969 UK Pressing) (score: 0.6410 - 64.10%)
3. Barbie 1985 Day to Night Edition (score: 0.5820 - 58.20%)

Query: antique silver coins pre 1900
1. 1904 Morgan Silver Dollar (score: 0.6801 - 68.01%)
2. Hot Wheels 1995 Treasure Hunt '57 Chevy (score: 0.5786 - 57.86%)
3. Miles Davis Kind of Blue 1959 Vinyl (score: 0.5741 - 57.41%)

Query: limited edition hot wheels cars
1. Hot Wheels 1995 Treasure Hunt '57 Chevy (score: 0.6977 - 69.77%)
2. Bar

**Normalization**

Trying to address the generalization meaning that gemini is using when embedding. Using softmax, min-max and z-score.

In [25]:
for i, query_embedding in enumerate(query_embeddings_np):
    user_query = user_queries[i]
    similarities = cosine_similarity([query_embedding], product_embeddings_np)[0]
    normalized_scores = softmax(similarities)

    top_n = 3
    top_indices = np.argsort(normalized_scores)[::-1][:top_n]

    print(f"\nQuery: {user_query}")
    for idx in top_indices:
        print(f"{product_listings[idx]['title']} (score: {normalized_scores[idx]:.4f})")


Query: batman comics from the 80s
Batman #404 Year One Part 1 1987 (score: 0.1174)
Barbie 1985 Day to Night Edition (score: 0.1013)
Charizard Holo 1st Edition (score: 0.0998)

Query: first edition pokemon cards
Charizard Holo 1st Edition (score: 0.1140)
MTG Black Lotus Proxy for Display (score: 0.1038)
Hot Wheels 1995 Treasure Hunt '57 Chevy (score: 0.1010)

Query: vintage vinyl jazz albums
Miles Davis Kind of Blue 1959 Vinyl (score: 0.1146)
The Beatles Abbey Road (1969 UK Pressing) (score: 0.1052)
Barbie 1985 Day to Night Edition (score: 0.0992)

Query: antique silver coins pre 1900
1904 Morgan Silver Dollar (score: 0.1112)
Hot Wheels 1995 Treasure Hunt '57 Chevy (score: 0.1005)
Miles Davis Kind of Blue 1959 Vinyl (score: 0.1000)

Query: limited edition hot wheels cars
Hot Wheels 1995 Treasure Hunt '57 Chevy (score: 0.1131)
Barbie 1985 Day to Night Edition (score: 0.1029)
Charizard Holo 1st Edition (score: 0.1025)

Query: signed baseball memorabilia
Derek Jeter Signed Yankees Jersey 

In [28]:
# use other normarlization techniques
# min-max
top_n = 3

for i, query_embedding in enumerate(query_embeddings_np):
    user_query = user_queries[i]
    similarities = cosine_similarity([query_embedding], product_embeddings_np)[0]

    #min-max normalization
    min_score = np.min(similarities)
    max_score = np.max(similarities)
    min_max_scores = (similarities - min_score) / (max_score - min_score + 1e-10)  # avoid division by zero

    #z-score normalization
    mean_score = np.mean(similarities)
    std_score = np.std(similarities)
    z_scores = (similarities - mean_score) / (std_score + 1e-10)

    top_indices_minmax = np.argsort(min_max_scores)[::-1][:top_n]
    top_indices_zscore = np.argsort(z_scores)[::-1][:top_n]

    print(f"\nQuery: {user_query}")
    print("\nTop Results (Min-Max Normalized):")
    for idx in top_indices_minmax:
        print(f"{product_listings[idx]['title']} (score: {min_max_scores[idx]:.4f})")

    print("\nTop Results (Z-Score Normalized):")
    for idx in top_indices_zscore:
        print(f"{product_listings[idx]['title']} (score: {z_scores[idx]:.4f})")



Query: batman comics from the 80s

Top Results (Min-Max Normalized):
Batman #404 Year One Part 1 1987 (score: 1.0000)
Barbie 1985 Day to Night Edition (score: 0.3039)
Charizard Holo 1st Edition (score: 0.2315)

Top Results (Z-Score Normalized):
Batman #404 Year One Part 1 1987 (score: 2.8399)
Barbie 1985 Day to Night Edition (score: 0.2649)
Charizard Holo 1st Edition (score: -0.0029)

Query: first edition pokemon cards

Top Results (Min-Max Normalized):
Charizard Holo 1st Edition (score: 1.0000)
MTG Black Lotus Proxy for Display (score: 0.4759)
Hot Wheels 1995 Treasure Hunt '57 Chevy (score: 0.3250)

Top Results (Z-Score Normalized):
Charizard Holo 1st Edition (score: 2.5466)
MTG Black Lotus Proxy for Display (score: 0.7408)
Hot Wheels 1995 Treasure Hunt '57 Chevy (score: 0.2209)

Query: vintage vinyl jazz albums

Top Results (Min-Max Normalized):
Miles Davis Kind of Blue 1959 Vinyl (score: 1.0000)
The Beatles Abbey Road (1969 UK Pressing) (score: 0.5360)
Barbie 1985 Day to Night Edit

*insert here a place to test a vector database*